# Malicious (D)DoS Flow Integration — CIC18 to CIC17

This notebook evaluates the integration of malicious **(D)DoS** network flows from **GenIDS-CIC18** into **GenIDS-CIC17** for cross-dataset IDS generalization experiments.

The integration procedure follows three main principles:

1. A percentage of malicious **(D)DoS** flows is selected from CIC18 and integrated into CIC17 before model training.
2. The integrated CIC18 flows are removed from the CIC18 evaluation dataset to prevent data leakage.
3. The same number of **(D)DoS** flows is removed from CIC17 before integration, preserving the dataset size and keeping the class distribution as consistent as possible.

The default configuration uses a **20% malicious flow integration rate**. The same notebook can be reused for 40%, 60%, and 80% by changing the `INTEGRATION_RATE` parameter.


## 1. Environment Setup


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    auc,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


## 2. Experiment Configuration

Adjust the paths below according to your local environment or repository structure.

The variable `DDOS_CLASS_VALUE` must match the value used in the `multiclass` column to represent the **(D)DoS** class after preprocessing. In the original notebook, class value `2` was used for the malicious **(D)DoS** class.


In [ ]:
RANDOM_STATE = 42
INTEGRATION_RATE = 0.20  # Change to 0.40, 0.60, or 0.80 for the remaining scenarios.
TEST_SIZE = 0.80

DATA_DIR = Path("../datasets")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CIC17_PATH = DATA_DIR / "GenIDS-CIC17.csv"
CIC18_PATH = DATA_DIR / "GenIDS-CIC18.csv"

LABEL_COLUMN = "multiclass"
TIME_COLUMN = "bidirectional_first_seen_ms"
SOURCE_COLUMN = "source_dataset"
DDOS_CLASS_VALUE = 2

print(f"Integration rate: {INTEGRATION_RATE:.0%}")
print(f"(D)DoS class value: {DDOS_CLASS_VALUE}")


## 3. Helper Functions


In [ ]:
def load_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load a dataset from a CSV file."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found for {dataset_name}: {path}. "
            "Update DATA_DIR or the file name in the configuration cell."
        )
    dataframe = pd.read_csv(path)
    print(f"{dataset_name}: {dataframe.shape}")
    return dataframe


def remove_non_model_columns(dataframe: pd.DataFrame, columns_to_remove: list[str]) -> pd.DataFrame:
    """Remove columns that should not be used during model training."""
    return dataframe.drop(columns=columns_to_remove, errors="ignore").copy()


def encode_categorical_features(
    dataframes: list[pd.DataFrame],
    categorical_columns: list[str],
) -> list[pd.DataFrame]:
    """Encode categorical feature columns independently for each dataset."""
    encoded_dataframes = []
    for dataframe in dataframes:
        dataframe = dataframe.copy()
        for column in categorical_columns:
            if column in dataframe.columns:
                dataframe[column] = LabelEncoder().fit_transform(dataframe[column].astype(str))
        encoded_dataframes.append(dataframe)
    return encoded_dataframes


def standardize_numeric_types(dataframe: pd.DataFrame, label_column: str) -> pd.DataFrame:
    """Convert numeric features to float while preserving the label column as integer."""
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if column == label_column:
            dataframe[column] = dataframe[column].astype(int)
        elif pd.api.types.is_numeric_dtype(dataframe[column]):
            dataframe[column] = dataframe[column].astype(float)
    return dataframe


def print_dataset_summary(dataframe: pd.DataFrame, dataset_name: str, label_column: str = LABEL_COLUMN) -> None:
    """Print dataset shape and class distribution."""
    print(f"\n{dataset_name}")
    print(f"Shape: {dataframe.shape}")
    if label_column in dataframe.columns:
        print("Class counts:")
        print(dataframe[label_column].value_counts().sort_index())
        print("Class distribution:")
        print(dataframe[label_column].value_counts(normalize=True).sort_index().map("{:.2%}".format))


def select_class_subset(
    dataframe: pd.DataFrame,
    class_value: int,
    integration_rate: float,
    label_column: str = LABEL_COLUMN,
    sort_column: str | None = TIME_COLUMN,
) -> pd.DataFrame:
    """Select a percentage of flows from a specific class."""
    class_flows = dataframe[dataframe[label_column] == class_value].copy()
    if sort_column in class_flows.columns:
        class_flows = class_flows.sort_values(by=sort_column)
    subset_size = int(len(class_flows) * integration_rate)
    return class_flows.iloc[:subset_size].copy()


def integrate_class_flows(
    target_dataframe: pd.DataFrame,
    source_dataframe: pd.DataFrame,
    class_value: int,
    integration_rate: float,
    target_name: str,
    source_name: str,
    class_name: str,
    label_column: str = LABEL_COLUMN,
    time_column: str = TIME_COLUMN,
    source_column: str = SOURCE_COLUMN,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Integrate class-specific source flows into the target dataset without data leakage.

    Returns:
        integrated_target: target dataset after removal and integration.
        source_test: source dataset after removing integrated flows.
        integrated_source_flows: source flows added to the target dataset.
        removed_target_flows: target flows removed to preserve dataset size.
    """
    integrated_source_flows = select_class_subset(
        source_dataframe,
        class_value=class_value,
        integration_rate=integration_rate,
        label_column=label_column,
        sort_column=time_column,
    )

    source_test = source_dataframe.drop(index=integrated_source_flows.index).copy()

    target_class_flows = target_dataframe[target_dataframe[label_column] == class_value].copy()
    if time_column in target_class_flows.columns:
        target_class_flows = target_class_flows.sort_values(by=time_column)

    removal_size = min(len(integrated_source_flows), len(target_class_flows))
    removed_target_flows = target_class_flows.iloc[:removal_size].copy()
    target_after_removal = target_dataframe.drop(index=removed_target_flows.index).copy()

    target_after_removal[source_column] = target_name.lower()
    integrated_source_flows[source_column] = f"{source_name.lower()}_{int(integration_rate * 100)}_{class_name.lower()}"

    integrated_target = pd.concat(
        [target_after_removal, integrated_source_flows],
        ignore_index=True,
    )

    if time_column in integrated_target.columns:
        integrated_target = integrated_target.sort_values(by=time_column).reset_index(drop=True)

    return integrated_target, source_test, integrated_source_flows, removed_target_flows


def split_and_scale(
    train_dataframe: pd.DataFrame,
    external_test_dataframe: pd.DataFrame,
    label_column: str = LABEL_COLUMN,
    source_column: str = SOURCE_COLUMN,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Split the training dataset and apply standard scaling to internal and external test sets."""
    train_dataframe = train_dataframe.drop(columns=[source_column], errors="ignore").copy()
    external_test_dataframe = external_test_dataframe.drop(columns=[source_column], errors="ignore").copy()

    X = train_dataframe.drop(columns=[label_column])
    y = train_dataframe[label_column]

    X_external = external_test_dataframe.drop(columns=[label_column])
    y_external = external_test_dataframe[label_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_external_scaled = scaler.transform(X_external)

    return X_train_scaled, X_test_scaled, X_external_scaled, y_train, y_test, y_external, X.columns


def build_xgboost_model(num_classes: int, random_state: int = RANDOM_STATE) -> XGBClassifier:
    """Create the XGBoost classifier used in the original experiment."""
    return XGBClassifier(
        eval_metric="mlogloss",
        n_estimators=300,
        max_depth=10,
        objective="multi:softprob",
        num_class=num_classes,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.5,
        random_state=random_state,
    )


def evaluate_model(model, X, y, dataset_name: str, class_labels: list[int]) -> dict:
    """Evaluate the model and print standard IDS metrics."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
    }

    try:
        y_bin = label_binarize(y, classes=class_labels)
        metrics["auc_roc_ovr"] = roc_auc_score(y_bin, y_proba, average="macro", multi_class="ovr")
        metrics["auc_pr_macro"] = average_precision_score(y_bin, y_proba, average="macro")
    except ValueError:
        metrics["auc_roc_ovr"] = np.nan
        metrics["auc_pr_macro"] = np.nan

    print(f"\nEvaluation: {dataset_name}")
    print("Confusion Matrix:")
    print(confusion_matrix(y, y_pred, labels=class_labels))
    print("\nClassification Report:")
    print(classification_report(y, y_pred, digits=4, zero_division=0))
    print("Metrics:")
    for key, value in metrics.items():
        if key != "dataset":
            print(f"{key}: {value:.4f}" if pd.notna(value) else f"{key}: NaN")

    return metrics


def plot_confusion_matrix(model, X, y, dataset_name: str, class_labels: list[int]) -> None:
    """Plot the confusion matrix."""
    ConfusionMatrixDisplay.from_estimator(
        model,
        X,
        y,
        labels=class_labels,
        values_format="d",
    )
    plt.title(f"Confusion Matrix — {dataset_name}")
    plt.tight_layout()
    plt.show()


def plot_multiclass_roc_curve(model, X, y, dataset_name: str, class_labels: list[int]) -> None:
    """Plot one-vs-rest ROC curves for a multiclass classifier."""
    y_bin = label_binarize(y, classes=class_labels)
    y_proba = model.predict_proba(X)

    plt.figure(figsize=(8, 6))
    for index, class_label in enumerate(class_labels):
        fpr, tpr, _ = roc_curve(y_bin[:, index], y_proba[:, index])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"Class {class_label} (AUC = {roc_auc:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {dataset_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


## 4. Load Datasets


In [ ]:
df_cic17_raw = load_dataset(CIC17_PATH, "GenIDS-CIC17")
df_cic18_raw = load_dataset(CIC18_PATH, "GenIDS-CIC18")


## 5. Preprocessing

Non-predictive identifiers and dataset-specific metadata columns are removed. Categorical features are encoded and numeric types are standardized.


In [ ]:
cic17_columns_to_remove = [
    "binary",
    "date",
    "hours",
    "expiration_id",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

cic18_columns_to_remove = [
    "binary",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

df_cic17 = remove_non_model_columns(df_cic17_raw, cic17_columns_to_remove)
df_cic18 = remove_non_model_columns(df_cic18_raw, cic18_columns_to_remove)

categorical_feature_columns = ["application_name", "application_category_name"]
df_cic17, df_cic18 = encode_categorical_features(
    [df_cic17, df_cic18],
    categorical_feature_columns,
)

df_cic17 = standardize_numeric_types(df_cic17, LABEL_COLUMN)
df_cic18 = standardize_numeric_types(df_cic18, LABEL_COLUMN)

print_dataset_summary(df_cic17, "GenIDS-CIC17")
print_dataset_summary(df_cic18, "GenIDS-CIC18")


## 6. Malicious (D)DoS Flow Integration

Malicious **(D)DoS** flows from CIC18 are integrated into CIC17. The selected CIC18 flows are removed from the CIC18 evaluation set to avoid data leakage.


In [ ]:
integrated_cic17, cic18_test, integrated_cic18_ddos, removed_cic17_ddos = integrate_class_flows(
    target_dataframe=df_cic17,
    source_dataframe=df_cic18,
    class_value=DDOS_CLASS_VALUE,
    integration_rate=INTEGRATION_RATE,
    target_name="CIC17",
    source_name="CIC18",
    class_name="ddos",
)

print_dataset_summary(integrated_cic17, "Integrated GenIDS-CIC17")
print_dataset_summary(cic18_test, "GenIDS-CIC18 after leakage-safe removal")

print(f"\nIntegrated CIC18 (D)DoS flows: {integrated_cic18_ddos.shape}")
print(f"Removed CIC17 (D)DoS flows: {removed_cic17_ddos.shape}")


## 7. Optional Visualization of Integrated Flows


In [ ]:
if TIME_COLUMN in integrated_cic17.columns and "bidirectional_bytes" in integrated_cic17.columns:
    plt.figure(figsize=(16, 8))

    target_flows = integrated_cic17[integrated_cic17[SOURCE_COLUMN] == "cic17"]
    integrated_flows = integrated_cic17[
        integrated_cic17[SOURCE_COLUMN] == f"cic18_{int(INTEGRATION_RATE * 100)}_ddos"
    ]

    plt.scatter(
        target_flows[TIME_COLUMN],
        target_flows["bidirectional_bytes"],
        label="CIC17 flows",
        alpha=0.5,
    )
    plt.scatter(
        integrated_flows[TIME_COLUMN],
        integrated_flows["bidirectional_bytes"],
        label="Integrated (D)DoS flows from CIC18",
        marker="x",
        alpha=0.8,
    )

    plt.title("Flow Volume over Time after Malicious Flow Integration")
    plt.xlabel("Timestamp (bidirectional_first_seen_ms)")
    plt.ylabel("Data Volume (bidirectional_bytes)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped because the required columns are not available.")


## 8. Train-Test Split and Normalization

The model is trained using the integrated CIC17 dataset. CIC18 is used as the external cross-dataset generalization test.


In [ ]:
X_train_scaled, X_test_scaled, X_cic18_scaled, y_train, y_test, y_cic18, feature_names = split_and_scale(
    train_dataframe=integrated_cic17,
    external_test_dataframe=cic18_test,
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Internal test set: {X_test_scaled.shape}")
print(f"External test set (CIC18): {X_cic18_scaled.shape}")


## 9. Model Training


In [ ]:
class_labels = sorted(pd.concat([y_train, y_test, y_cic18]).unique().tolist())
num_classes = len(class_labels)

model = build_xgboost_model(num_classes=num_classes, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

print("Model training completed.")
print(f"Class labels: {class_labels}")


## 10. Feature Importance


In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_,
}).sort_values(by="importance", ascending=False)

feature_importance.head(20)


## 11. Internal Evaluation — Integrated CIC17


In [ ]:
internal_metrics = evaluate_model(
    model=model,
    X=X_test_scaled,
    y=y_test,
    dataset_name="Integrated CIC17 internal test",
    class_labels=class_labels,
)

plot_confusion_matrix(
    model=model,
    X=X_test_scaled,
    y=y_test,
    dataset_name="Integrated CIC17 internal test",
    class_labels=class_labels,
)

plot_multiclass_roc_curve(
    model=model,
    X=X_test_scaled,
    y=y_test,
    dataset_name="Integrated CIC17 internal test",
    class_labels=class_labels,
)


## 12. Cross-Dataset Generalization Evaluation — CIC18


In [ ]:
external_metrics = evaluate_model(
    model=model,
    X=X_cic18_scaled,
    y=y_cic18,
    dataset_name="CIC18 generalization test",
    class_labels=class_labels,
)

plot_confusion_matrix(
    model=model,
    X=X_cic18_scaled,
    y=y_cic18,
    dataset_name="CIC18 generalization test",
    class_labels=class_labels,
)

plot_multiclass_roc_curve(
    model=model,
    X=X_cic18_scaled,
    y=y_cic18,
    dataset_name="CIC18 generalization test",
    class_labels=class_labels,
)


## 13. Save Metrics


In [ ]:
metrics = pd.DataFrame([internal_metrics, external_metrics])
metrics_path = RESULTS_DIR / f"malicious_flow_integration_{int(INTEGRATION_RATE * 100)}pct_cic18_to_cic17_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print(f"Metrics saved to: {metrics_path}")
metrics
